In [1]:
import config
import pandas as pd
import modules.capiq as capiq
import modules.gpt as gpt
import textwrap
import modules.sql as sql
import json
import modules.utils as utils

In [2]:
# Load the JSON file
with open(f'{config.ROOT}/data/WRDS_ECs_companies_test.json', 'r') as file:
    test_companies = json.load(file)

# Get list of test company ids from the JSON file
test_company_ids = list(test_companies.keys())

# Load capiq dataset
df = pd.read_feather(f'{config.ROOT}/data/transcript-detail.feather')

In [3]:
# Get transcripts for the list of test company ids
test_transcripts_df = pd.DataFrame()
for company_id in test_company_ids:
    company_transcripts = df[df['companyid'] == int(company_id)]
    # Keep only the most recent version of each transcript
    company_transcripts = utils.keep_most_recent_transcripts(company_transcripts)
    test_transcripts_df = pd.concat([test_transcripts_df, company_transcripts])

# Sort the transcripts by companyid and transcript id
test_transcripts_df = test_transcripts_df.sort_values(by=['companyid', 'transcriptid'])

/Users/ioanrusu/Desktop/collusion prediction project/collusion-llm/modules/utils.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['transcriptcreationdate_utc'] = df['transcriptcreationdate_utc'].astype(str)
/Users/ioanrusu/Desktop/collusion prediction project/collusion-llm/modules/utils.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['transcriptcreationtime_utc'] = df['transcriptcreationtime_utc'].astype(str)
/Users/ioanrusu/Desktop/collusion prediction project/collusion-llm/modules/utils.py:

In [4]:
# Add fiscal year and fiscal quarter columns
test_transcripts_df[['fiscal_quarter', 'fiscal_year']] = test_transcripts_df['headline'].apply(
    lambda headline: pd.Series(utils.get_quarter_year_from_headline(headline))
)

In [5]:
# Add the airlines capacity indicators
# Load matched_airlines_test.xlsx
matched_airlines_df = pd.read_excel(f'{config.ROOT}/data/matched_airlines_test.xlsx')

# Merge the dataframes on companyid, fiscal_year, and fiscal_quarter
test_transcripts_df = test_transcripts_df.merge(
    matched_airlines_df[['companyid', 'fiscal_year', 'fiscal_quarter', 'auto_capacity', 'manual_capacity']],
    how='left',
    on=['companyid', 'fiscal_year', 'fiscal_quarter']
)


In [6]:
# Add the uhaul joe score
# Load uhaul_joe_scores_test.xlsx
uhaul_joe_scores_df = pd.read_excel(f'{config.ROOT}/data/uhaul_joe_score.xlsx')

# Merge the dataframes on companyid and transcriptid
test_transcripts_df = test_transcripts_df.merge(
    uhaul_joe_scores_df[['companyid', 'transcriptid', 'joe_score', 'joe_comment']],
    how='left',
    on=['companyid', 'transcriptid']
)


In [7]:
# Load the JSON data
with open(f'{config.ROOT}/data/examples_literature.json') as f:
    examples_literature = json.load(f)

# Add a new column 'is_example' and initialize it to false
test_transcripts_df['is_example'] = 0

# Iterate through each company in the JSON data
for companyid, company_data in examples_literature.items():
    # Iterate through each call for the company
    for call in company_data['calls']:
        fiscal_year = call['fiscal_year']
        fiscal_quarter = call['fiscal_quarter']
        
        # Set 'is_example' to 1 for matching rows in test_transcripts_df
        test_transcripts_df.loc[
            (test_transcripts_df['companyid'] == int(companyid)) &
            (test_transcripts_df['fiscal_year'] == fiscal_year) &
            (test_transcripts_df['fiscal_quarter'] == fiscal_quarter),
            'is_example'
        ] = 1


In [8]:
# Save test_transcripts_df to an excel file 
test_transcripts_df.to_excel(f'{config.ROOT}/data/FullTestSet.xlsx', index=False)